In [1]:
import os
import pathlib

here = pathlib.Path.cwd()

ROOT = here.parents[2] if here.name == "day04" else here

os.chdir(ROOT)

SANDBOX = ROOT / "sandbox" / "w3" / "day04"

print("프로젝트 루트  :", ROOT)

프로젝트 루트  : c:\workspace\hanwha-agent


- 기본 설정

In [2]:
import anthropic
import sys


print("Anthropic 버전 : ", anthropic.__version__)
print("지금 커널의 파이썬 경로 : ", sys.executable)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

Anthropic 버전 :  1.6.0
지금 커널의 파이썬 경로 :  c:\workspace\hanwha-agent\.venv\Scripts\python.exe


In [3]:
from app.core.config import get_settings, mask

settings = get_settings()
print('모드        :', settings.app_mode)
print('모델        :', settings.llm_model)

if settings.anthropic_api_key is None:
    print('API 키      : (없음) — .env 의 ANTHROPIC_API_KEY 가 비어 있습니다')
else:
    print('API 키      :', mask(settings.anthropic_api_key.get_secret_value()))

모드        : mock
모델        : claude-haiku-4-5
API 키      : sk-ant-a...(108자)


In [4]:
import inspect

from anthropic.resources.messages import Messages

# Messages.create 
# - API한테 보내줄 메시지 작성하는 기능 
params = inspect.signature(Messages.create).parameters
# 필수 파라미터들만 조회 
required = [name for name, p in params.items() 
            if p.default is inspect.Parameter.empty and name != "self"]
print(required)

QUESTION = "제주도 출장 숙박비 한도가 얼마인가요?"

try:
    Messages.create(
        None,  # client 와야하는 자리 
        model="claude-haiku-4-5", 
        # max_tokens
        messages=[
            {"role": "user", "content": QUESTION}
        ]
    )
except TypeError as exc:
    print(type(exc).__name__)
    print(exc)

['max_tokens', 'messages', 'model']
TypeError
Missing required arguments; Expected either ('max_tokens', 'messages' and 'model') or ('max_tokens', 'messages', 'model' and 'stream') arguments to be given


* 금액 계산

In [5]:
PRICING = {'input': 1.0, 'cache_write': 1.25, 'cache_read': 0.1, 'output': 5.0}
USD_KRW = 1400.0

total_input = 67
output_tok = 236

def cost_krw(input_tok: int, output_tok: int) -> float:
    usd = (input_tok * PRICING['input'] + output_tok * PRICING['output']) / 1000000
    return round(usd * USD_KRW, 1)

(WED_INPUT, WED_OUTPUT) = (800, 400)
wed = cost_krw(WED_INPUT, WED_OUTPUT) # 가짜 금액
today = cost_krw(total_input, output_tok) # 진짜 금액
print(f'수요일에 어림한 값 : {wed } 원   (입력 {WED_INPUT} · 출력 {WED_OUTPUT} 토큰)')
print(f'오늘 실제 호출     : {today} 원   (입력 {total_input} · 출력 {output_tok} 토큰)')
print(f'차이               : {round(wed - today, 1)} 원')

수요일에 어림한 값 : 3.9 원   (입력 800 · 출력 400 토큰)
오늘 실제 호출     : 1.7 원   (입력 67 · 출력 236 토큰)
차이               : 2.2 원


- 포트 확인

In [8]:
# importlib.invalidate_caches()
# - 파이썬은 한 번 훑은 폴더의 파일 목록을 기억해둔다. 
# - 새로 만든 파일을 확인하지 못할 수 있다.
# - 캐시 삭제 후 다시 훑게 하기
import importlib
importlib.invalidate_caches()

from app.integrations.ports import LLMPort, LLMResult

result = LLMResult(text="1박 70,000원 이내", model="claude-haiku-4-5")
print(result)
print("기본값 확인")
print("input_token: ", result.input_tok)
print("extras: ", result.extras)
print("cost_kor: ", result.cost_krw)
print("LLMPort: ", LLMPort.__name__)

LLMResult(text='1박 70,000원 이내', model='claude-haiku-4-5', input_tok=0, cache_tok=0, output_tok=0, cost_krw=0.0, latency_ms=0, extras={})
기본값 확인
input_token:  0
extras:  {}
cost_kor:  0.0
LLMPort:  LLMPort


In [ ]:
import importlib
importlib.invalidate_caches()

from app.integrations.llm_claude import USD_KRW, estimate_cost_krw

input_tok = 67
output_tok = 236

print("환율: ", USD_KRW)
print("원화로 계산한 비용 : ", estimate_cost_krw(input_tok, output_tok), "원")

환율:  1400.0
원화로 계산한 비용 :  1.7 원


In [11]:
import importlib
import inspect
from app.integrations import llm_claude

llm_claude = importlib.reload(llm_claude)

lines = inspect.getsource(llm_claude.ClaudeLLM.__init__).splitlines()

(shown, inside_doc) = ([], False)

for line in lines:
    if line.strip().startswith('"""'):
        inside_doc = not inside_doc
        continue
    if not inside_doc:
        shown.append(line)
        
print('── ClaudeLLM.__init__ 소스 (docstring 은 접었다) ──')
print('\n'.join(shown))
print('answer 있는가:', hasattr(llm_claude.ClaudeLLM, 'answer'))


── ClaudeLLM.__init__ 소스 (docstring 은 접었다) ──
    def __init__(self) -> None:
        try:
            from anthropic import Anthropic
        except ImportError as exc:
            raise ExternalServiceError("Anthropic 패키지가 설치되어 있지 않습니다.") from exc

        settings = get_settings()
        key = settings.anthropic_api_key
        if key is None:
            raise ExternalServiceError("ANTHROPIC_API_KEY 값이 비어있습니다.")
        self._client = Anthropic(api_key=key.get_secret_value())
        self._model = settings.llm_model
answer 있는가: False


In [13]:
import importlib
import inspect
from app.integrations import llm_claude, ports
llm_claude = importlib.reload(llm_claude)
sig_port = inspect.signature(ports.LLMPort.answer, eval_str=True)
sig_impl = inspect.signature(llm_claude.ClaudeLLM.answer, eval_str=True)
print('LLMPort.answer   :', sig_port)
print('ClaudeLLM.answer :', sig_impl)
print()
print('_load_prompt 있는가   :', hasattr(llm_claude, '_load_prompt'), '  ← 추후에 만든다')
print('_context_block 있는가 :', hasattr(llm_claude, '_context_block'), '  ← 추후에 만든다')

LLMPort.answer   : (self, *, question: str, contexts: list[dict], user: dict) -> app.integrations.ports.LLMResult
ClaudeLLM.answer : (self, *, question: str, contexts: list[dict], usser: dict) -> app.integrations.ports.LLMResult

_load_prompt 있는가   : False   ← 추후에 만든다
_context_block 있는가 : False   ← 추후에 만든다


- 팩토리
    - live인지 확인

In [17]:
import importlib
importlib.invalidate_caches()

from app.core.config import get_settings
from app.core.exceptions import ModeNotAvailable
from app.integrations.factory import get_llm

print('is_live :', get_settings().is_live)
try:
    llm = get_llm()
    print('어댑터 :', llm.name)
except ModeNotAvailable as exc:
    print(f'{type(exc).__name__} ({exc.status_code})')
    print(exc.message)

is_live : False
ModeNotAvailable (409)
테스트용 mock 어댑터는 만들지 않았습니다..env의 APP_MODE의 값이 live인지 확인해주세요.


In [18]:
from pathlib import Path

P = Path("backend") / "app" / "agent" / "prompts" / "answer_system.md"
text = P.read_text(encoding="utf-8")

print("존재 여부 : ", P.exists())
print("첫 2줄 : ", text.splitlines()[:2])
print("글자 수 : ", len(text))

존재 여부 :  True
첫 2줄 :  ['## 역할', '']
글자 수 :  812


In [ ]:
import importlib
import sys
from pathlib import Path

BACKEND = Path.cwd() / 'backend'
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
    
import app.integrations.llm_claude as llm_claude
importlib.reload(llm_claude)

from app.integrations.llm_claude import PROMPTS, _load_prompt

print('PROMPTS 경로 :', PROMPTS)

good = _load_prompt('answer_system.md')
# 조용한 에러 
# - 오타 하나가 이렇게 위험하다
bad = _load_prompt('answer_sytem.md')

print('정상 이름 : 글자 수', len(good), '->', repr(good.splitlines()[0]))
print('오타 이름 : 글자 수', len(bad), '  ->', repr(bad))


PROMPTS 경로 : C:\workspace\hanwha-agent\backend\app\agent\prompts
정상 이름 : 글자 수 812 -> '## 역할'
오타 이름 : 글자 수 0   -> ''


In [20]:
CONTEXTS = [
    {
        "doc_id": "DOC-HR-014",
        "title": "국내출장 여비 규정",
        "version": "v2.0",
        "locator": "제12조(숙박비) · p.6",
        "quote": "서울·광역시: 1박 70,000원 이내",
        "score": 0.91,
    },
    {
        "doc_id": "DOC-PU-007",
        "title": "구매·계약 규정",
        "version": "v4.0",
        "locator": "제7조 · p.3",
        "quote": "출장 중 물품 구매는 사전 품의를 원칙으로 한다.",
        "score": 0.62,
    },
    {
        "doc_id": "DOC-SE-003",
        "title": "정보보안 지침",
        "version": "v2.2",
        "locator": "제4조 · p.2",
        "quote": "대외비 문서는 열람 권한이 확인된 임직원에게만 제공한다.",
        "score": 0.48,
    },
]


In [21]:
def context_block(contexts: list[dict]) -> str:
    lines = []
    for i, c in enumerate(contexts, 1):
        lines.append(
            f"[근거 {i}] {c.get('title')} {c.get('version')} · {c.get('locator')} "
            f"(유사도 {c.get('score', 0):.2f})\n{c.get('quote') or c.get('text') or ''}"
        )
    return "\n\n".join(lines) if lines else "(근거 문서 없음)"


print(context_block(CONTEXTS))
print()
print("── 근거가 0건일 때 ──")
print(context_block([]))

[근거 1] 국내출장 여비 규정 v2.0 · 제12조(숙박비) · p.6 (유사도 0.91)
서울·광역시: 1박 70,000원 이내

[근거 2] 구매·계약 규정 v4.0 · 제7조 · p.3 (유사도 0.62)
출장 중 물품 구매는 사전 품의를 원칙으로 한다.

[근거 3] 정보보안 지침 v2.2 · 제4조 · p.2 (유사도 0.48)
대외비 문서는 열람 권한이 확인된 임직원에게만 제공한다.

── 근거가 0건일 때 ──
(근거 문서 없음)


In [ ]:
import importlib
import app.integrations.llm_claude as llm_claude
importlib.reload(llm_claude)

from app.integrations.llm_claude import _extract_json
from app.schemas.chat import AnswerOut
from pydantic import ValidationError

# 모델이 json 형식의 응답을 ``` 백틱으로 감싸서 주는 경우 (예시)
RAW = (
    "```json\n"
    '{"answer":"1박 70,000원 이내",'
    '"sources":[{"doc_id":"DOC-HR-014","title":"국내출장 여비 규정",'
    '"version":"v2.0","locator":"제12조(숙박비) · p.6"}],'
    '"enough_evidence":true}\n'
    "```"
)

# 모델이 답변한 그대로 원문을 체크하면
try:
    # 백틱 때문에 에러 발생
    AnswerOut.model_validate_json(RAW)
except ValidationError as e:
    print('① 그대로 넣으면 :', e.errors()[0]['msg'])

# 모델이 답변한 json을 전처리하고
data = _extract_json(RAW)
# 체크하면
a = AnswerOut.model_validate(data)

print(f'② 벗기고 검증   : answer={a.answer!r} · 근거 {len(a.sources)}건 · 충분함 {a.enough_evidence}')
print('③ JSON 이 아니면 :', _extract_json('죄송합니다. 답변을 만들지 못했습니다.'))


① 그대로 넣으면 : Invalid JSON: expected value at line 1 column 1
② 벗기고 검증   : answer='1박 70,000원 이내' · 근거 1건 · 충분함 True
③ JSON 이 아니면 : {}


: 